### Mount Google Drive for Saving Results

Run this cell to connect your Google Drive. Everything is saved to a folder named `Page8_Best_of_N` in your Drive. Outside Colab it goes to a local folder of the same name.

Progress is saved as it goes, so a run that disconnects picks up where it stopped when the cells are run again. For a fresh run, empty the folder first.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/Page8_Best_of_N'
except ImportError:
    SAVE_DIR = 'Page8_Best_of_N'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to {SAVE_DIR}")

### Step 1: Install Dependencies, Imports and Configurations

First, we install the required libraries, import the packages, and set the run's parameters: the four language models that judge by answering Yes or No, the two reward models that only judge, and the sampling settings for the 3B's 16 chains.

In [ ]:
#@title Installation and Setup
!pip install -q datasets transformers accelerate math-verify

import copy
import gc
import json
import math
import os
import random
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from math_verify import parse, verify
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Qwen2Config, Qwen2Model

# Language models: each judges the 3B's chains by answering Yes or No
GENERATORS = [
    "Qwen/Qwen2.5-Math-1.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]
# Reward models: they score a chain but cannot write one
REWARD_MODELS = [
    "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B",
    "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B",
]
WRITER = "Qwen/Qwen2.5-3B-Instruct"                     # writes the 16 chains
VERIFIER = "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B"  # the trained verifier in steps 10 and 11

N_CHAINS = 16
TEMPERATURE = 0.3
TOP_P = 0.95
MAX_NEW_TOKENS = 2048

BATCH_SIZE = 128            # sequences per generate call; lower it if memory runs out
TOKENS_PER_BATCH = 32_768   # padded tokens per scoring pass
N_SUBSETS = 200             # random subsets of the 16 chains behind each point in step 10

N_PROBLEMS = 100            # drawn from the 500, in proportion to their levels

SEED = 42
DEVICE = "cuda"
random.seed(SEED)
torch.manual_seed(SEED)


def short(name):
    return name.split("/")[-1]


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def saved(name):
    return os.path.join(SAVE_DIR, name)


def save_json(path, obj):
    with open(path + ".tmp", "w") as f:
        json.dump(obj, f)
    os.replace(path + ".tmp", path)

### Step 2: Load the MATH-500 Dataset

We pull `HuggingFaceH4/MATH-500` and draw 100 of its problems at random, the same share from each level as in the full 500. The seed and the draw are the frequency-voting notebook's, so these are the same 100 problems.

In [ ]:
print("Loading MATH-500 dataset...")
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")

# The same share of each level as the 500, drawn at random within each level, kept in
# dataset order.
by_level = {}
for k, row in enumerate(dataset):
    by_level.setdefault(row["level"], []).append(k)
draw = random.Random(SEED)
quota = {lv: round(len(ks) * N_PROBLEMS / len(dataset)) for lv, ks in by_level.items()}
quota[max(quota, key=lambda lv: len(by_level[lv]))] += N_PROBLEMS - sum(quota.values())
picked = sorted(k for lv, ks in by_level.items() for k in draw.sample(ks, quota[lv]))
problems = [dataset[k] for k in picked]

full = Counter(row["level"] for row in dataset)
levels = Counter(p["level"] for p in problems)
print(f"{len(problems)} of {len(dataset)} problems, by level: "
      + ", ".join(f"{lv}: {levels[lv]} of {full[lv]}" for lv in sorted(levels)))

### Step 3: Define the Prompt, Answer Extraction and Grading

Every model solves with the same chain-of-thought prompt and writes its final answer inside `\boxed{}`. The answer is the contents of the last box. Two answers are the same answer when they match as text or when `math_verify` finds them mathematically equal (`\frac{1}{2}` and `0.5`). The same test grades a chain against the reference answer and groups the 16 chains' answers for the vote.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant. Solve the math problem step-by-step, explaining your reasoning. Put your final answer inside \\boxed{answer}."


def build_prompt(tok, problem):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_answer(text):
    """The contents of the last \\boxed{...}, with nested braces matched. None if there is no box."""
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    i = start + len("\\boxed{")
    depth = 1
    for j in range(i, len(text)):
        if text[j] == "{":
            depth += 1
        elif text[j] == "}":
            depth -= 1
            if depth == 0:
                return text[i:j].strip() or None
    return None


_parsed, _same = {}, {}


def parsed(answer):
    if answer not in _parsed:
        _parsed[answer] = parse("\\boxed{" + answer + "}")
    return _parsed[answer]


def same_answer(a, b):
    if a is None or b is None:
        return False
    if a.replace(" ", "") == b.replace(" ", ""):
        return True
    if (a, b) not in _same:
        _same[(a, b)] = bool(verify(parsed(a), parsed(b)))
    return _same[(a, b)]


def grade(texts, gold):
    """One row per chain: its text, its answer, and whether the answer is right."""
    rows = []
    for text in texts:
        answer = extract_answer(text)
        rows.append({"text": text, "answer": answer, "correct": same_answer(gold, answer)})
    return rows


print(same_answer("\\frac{1}{2}", "0.5"), same_answer("\\dfrac{14}{3}", "\\frac{14}{3}"), same_answer("3", "4"))

### Step 4: Generate Chains

One helper writes every chain on this page. Greedy (T = 0) when `temperature` is 0, otherwise sampled at `temperature` with `top_p`. The repetition penalty and top-k that ship in Qwen's generation config are switched off, so the settings above are the only ones in play.

In [ ]:
def load_generator(name):
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map=DEVICE).eval()
    return model, tok


@torch.no_grad()
def generate_chains(model, tok, problems, n=1, temperature=0.0, checkpoint=None):
    """n chains per problem, returned as one list of n texts per problem."""
    config = copy.deepcopy(model.generation_config)
    config.update(
        max_new_tokens=MAX_NEW_TOKENS,
        num_return_sequences=n,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else None,
        top_p=TOP_P if temperature > 0 else None,
        top_k=None,
        repetition_penalty=1.0,
        pad_token_id=tok.pad_token_id,
    )
    per_call = max(1, BATCH_SIZE // n)
    chains = []
    if checkpoint and os.path.exists(checkpoint):
        with open(checkpoint) as f:
            chains = json.load(f)
        print(f"    Resuming at {len(chains)}/{len(problems)} problems.")
    for i in range(len(chains), len(problems), per_call):
        batch = problems[i : i + per_call]
        prompts = [build_prompt(tok, p["problem"]) for p in batch]
        inputs = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        outputs = model.generate(**inputs, generation_config=config)
        texts = tok.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        chains += [texts[k * n : (k + 1) * n] for k in range(len(batch))]
        if checkpoint:
            save_json(checkpoint, chains)
        print(f"    Progress: {len(chains)}/{len(problems)} problems.")
    return chains

### Step 5: One Greedy Chain from the 3B

Qwen2.5-3B-Instruct writes one greedy chain per problem. This is the one-chain baseline every rule below is compared with.

In [ ]:
print(f"Solving with {WRITER}, one greedy chain per problem")
start = time.time()
model, tok = load_generator(WRITER)
texts = generate_chains(model, tok, problems, checkpoint=saved(f"page8_greedy_texts_n{len(problems)}.json"))
greedy = [grade(t, p["answer"])[0] for t, p in zip(texts, problems)]
del model, tok
free_gpu()
ONE_CHAIN = np.array([row["correct"] for row in greedy])
print(f"  solved {ONE_CHAIN.mean():.1%} in {time.time() - start:.0f}s")

save_json(saved(f"page8_greedy_n{len(problems)}.json"), greedy)

### Step 6: The 3B Writes 16 Chains per Problem

Qwen2.5-3B-Instruct solves the 100 problems again, this time writing 16 chains per problem at T = 0.3 with the same prompt and the same 2,048-token cap. These 1,600 chains are the pile every rule below chooses from.

In [ ]:
print(f"Writing {N_CHAINS} chains per problem with {WRITER} at T = {TEMPERATURE}")
start = time.time()
torch.manual_seed(SEED)
model, tok = load_generator(WRITER)
texts = generate_chains(model, tok, problems, n=N_CHAINS, temperature=TEMPERATURE,
                        checkpoint=saved(f"page8_chain_texts_T{TEMPERATURE}_n{len(problems)}.json"))
chains = [grade(t, p["answer"]) for t, p in zip(texts, problems)]
del model, tok
free_gpu()
print(f"Done in {time.time() - start:.0f}s")

save_json(saved(f"page8_chains_T{TEMPERATURE}_n{len(problems)}.json"), chains)

C = np.array([[c["correct"] for c in row] for row in chains])   # (problems, chains): is each chain right?
print(f"one sampled chain is right {C.mean():.1%} of the time")
print(f"a right answer is among the 16 chains on {C.any(axis=1).mean():.1%} of problems")

### Step 7: Majority Vote

For each problem, the 16 final answers are grouped by equivalence and the most common group is kept. A chain with no boxed answer does not vote, and a tie for most common counts as wrong.

In [ ]:
def group_answers(answers):
    """The distinct answers, and for each chain the index of its group (None if it gave no answer)."""
    reps, labels = [], []
    for answer in answers:
        if answer is None:
            labels.append(None)
            continue
        for k, rep in enumerate(reps):
            if same_answer(rep, answer):
                labels.append(k)
                break
        else:
            reps.append(answer)
            labels.append(len(reps) - 1)
    return reps, labels


def vote_winner(labels):
    """The group with the most chains, or None when there is a tie or no answer at all."""
    ranked = Counter(l for l in labels if l is not None).most_common(2)
    if not ranked or (len(ranked) == 2 and ranked[0][1] == ranked[1][1]):
        return None
    return ranked[0][0]


groups = []
for row, p in zip(chains, problems):
    reps, labels = group_answers([c["answer"] for c in row])
    groups.append({"reps": reps, "labels": labels,
                   "rep_correct": [same_answer(p["answer"], r) for r in reps]})

winners = [vote_winner(g["labels"]) for g in groups]
VOTE = np.array([w is not None and g["rep_correct"][w] for w, g in zip(winners, groups)])
CEILING = C.any(axis=1)

print(f"majority vote solves {VOTE.mean():.1%}")
print(f"ties: {sum(w is None for w in winners)} problems")
print(f"ceiling (a right answer is among the chains): {CEILING.mean():.1%}")

### Step 8: Every Judge Scores Every Chain

No judge sees the reference answer, and each scores one chain at a time.

- **Language models** read the question and the chain and are asked "Is the final answer correct? Yes or No". The score is log P(Yes) − log P(No) at the position where the reply would start: one forward pass, nothing generated.
- **Reward models** read the question and the chain in Skywork's own format (each line of the chain closed by a newline token) and score the chain's last token. Skywork's checkpoint is a stock Qwen2 body plus a one-output value head, so the body loads into transformers' own `Qwen2Model` and the head is applied here; none of the checkpoint's custom code runs. The score is the sigmoid of the head.

Chains are batched by length, so each forward pass pads as little as possible.

In [ ]:
JUDGE_PROMPT = (
    "You are grading a proposed solution to a mathematics problem.\n\n"
    "Problem:\n{question}\n\n"
    "Proposed solution:\n{solution}\n\n"
    "Is the final answer correct? Yes or No"
)


def length_batches(lengths):
    """Indices grouped longest first, each group under TOKENS_PER_BATCH padded tokens."""
    order = sorted(range(len(lengths)), key=lambda i: -lengths[i])
    batch = []
    for i in order:
        if batch and lengths[batch[0]] * (len(batch) + 1) > TOKENS_PER_BATCH:
            yield batch
            batch = []
        batch.append(i)
    if batch:
        yield batch


@torch.no_grad()
def score_last_token(body, pad_id, sequences, read):
    """Runs each token sequence through `body` and applies `read` to the hidden state at its last token."""
    scores = np.empty(len(sequences), dtype=np.float32)
    batches = list(length_batches([len(s) for s in sequences]))
    for batch in tqdm(batches, desc="  scoring"):
        width = len(sequences[batch[0]])
        input_ids = torch.full((len(batch), width), pad_id)
        mask = torch.zeros((len(batch), width), dtype=torch.long)
        for row, i in enumerate(batch):
            seq = sequences[i]
            input_ids[row, width - len(seq):] = torch.tensor(seq)
            mask[row, width - len(seq):] = 1
        positions = (mask.cumsum(-1) - 1).clamp(min=0)
        hidden = body(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE),
                      position_ids=positions.to(DEVICE), use_cache=False).last_hidden_state[:, -1]
        scores[batch] = read(hidden).float().cpu().numpy()
    return scores


def judge_sequence(tok, question, solution):
    messages = [{"role": "user", "content": JUDGE_PROMPT.format(question=question, solution=solution)}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tok(text, add_special_tokens=False).input_ids


def load_reward_model(name):
    """The Qwen2 body in transformers' own implementation, plus the checkpoint's value head."""
    path = snapshot_download(name)
    state = torch.load(f"{path}/pytorch_model.bin", map_location="cpu", weights_only=True, mmap=True)
    config = Qwen2Config.from_pretrained(path)
    config._attn_implementation = "sdpa"
    torch.set_default_dtype(torch.bfloat16)
    with torch.device(DEVICE):
        body = Qwen2Model(config)
    torch.set_default_dtype(torch.float32)
    body.load_state_dict({k[len("model."):]: v for k, v in state.items() if k.startswith("model.")})
    w = state["v_head.summary.weight"].to(DEVICE, torch.float32).flatten()
    b = state["v_head.summary.bias"].to(DEVICE, torch.float32)
    tok = AutoTokenizer.from_pretrained(path)
    return body.eval(), w, b, tok


def reward_sequence(tok, question, solution):
    """Skywork's format: the question on its own line, then every line of the chain closed by a newline token."""
    newline = tok.encode("\n", add_special_tokens=False)[-1]
    ids = tok.encode(tok.bos_token + question + "\n", add_special_tokens=False)
    for step in solution.split("\n"):
        ids += (tok.encode(step, add_special_tokens=False) if step else []) + [newline]
    return ids


pairs = [(p["problem"], c["text"]) for p, row in zip(problems, chains) for c in row]
SCORES = {}


def scores_file(name):
    return saved(f"page8_scores_{short(name)}_T{TEMPERATURE}_n{len(problems)}.npy")


def load_scores(name):
    """A judge already scored by this run, loaded from Drive. False if it still has to run."""
    if not os.path.exists(scores_file(name)):
        return False
    SCORES[short(name)] = np.load(scores_file(name))
    print(f"\n{name}: already scored, loaded")
    return True


def save_scores(name):
    tmp = scores_file(name) + ".tmp.npy"
    np.save(tmp, SCORES[short(name)])
    os.replace(tmp, scores_file(name))

for name in GENERATORS:
    if load_scores(name):
        continue
    print(f"\nJudging with {name}")
    start = time.time()
    model, tok = load_generator(name)
    yes, no = (tok.encode(w, add_special_tokens=False) for w in ("Yes", "No"))
    assert len(yes) == len(no) == 1, "Yes and No must each be one token"

    def read(hidden):
        logp = torch.log_softmax(model.lm_head(hidden).float(), dim=-1)
        return logp[:, yes[0]] - logp[:, no[0]]

    sequences = [judge_sequence(tok, q, s) for q, s in pairs]
    SCORES[short(name)] = score_last_token(model.model, tok.pad_token_id, sequences, read).reshape(C.shape)
    save_scores(name)
    del model, tok, read
    free_gpu()
    print(f"  done in {time.time() - start:.0f}s")

for name in REWARD_MODELS:
    if load_scores(name):
        continue
    print(f"\nJudging with {name}")
    start = time.time()
    body, w, b, tok = load_reward_model(name)
    sequences = [reward_sequence(tok, q, s) for q, s in pairs]
    read = lambda hidden: torch.sigmoid(hidden.float() @ w + b)
    SCORES[short(name)] = score_last_token(body, tok.pad_token_id, sequences, read).reshape(C.shape)
    save_scores(name)
    del body, w, b, tok, read
    free_gpu()
    print(f"  done in {time.time() - start:.0f}s")

save_json(saved(f"page8_scores_T{TEMPERATURE}_n{len(problems)}.json"), {judge: s.tolist() for judge, s in SCORES.items()})

### Step 9: Best-of-N, Once per Judge

Each judge keeps its top-scored chain of the 16, and the problem counts as solved when that chain is right. Every judge chooses from the same chains, so the only thing that changes from row to row is the judge.

In [ ]:
JUDGES = [short(m) for m in GENERATORS + REWARD_MODELS]


def best_of(scores):
    """Whether the top-scored chain of each problem is right."""
    return C[np.arange(len(C)), scores.argmax(axis=1)]


BEST = {j: best_of(SCORES[j]) for j in JUDGES}

summary = pd.DataFrame({
    "Judge": JUDGES,
    "Best-of-16 (%)": [100 * BEST[j].mean() for j in JUDGES],
    "Beats the vote": [BEST[j].mean() > VOTE.mean() for j in JUDGES],
}).sort_values("Best-of-16 (%)", ascending=False).round(1)
display(summary)
print(f"3B, one greedy chain: {100 * ONE_CHAIN.mean():.1f}%")
print(f"majority vote over 16: {100 * VOTE.mean():.1f}%")
print(f"ceiling: {100 * CEILING.mean():.1f}%")

### Step 10: One Problem Where the Vote and the Verifier Disagree

One row per distinct answer among the 16 chains, most common at the top, and one dot per chain placed at the trained verifier's score. Green is a right chain, blue a wrong one, and the orange ring marks the chain the verifier keeps. The problem shown is drawn at random from those where the verifier picks a different answer from the vote and only the verifier is right; set `SPECIMEN` to show another.

In [ ]:
GREEN, BLUE, ORANGE, GRAY = "#1baf7a", "#2a78d6", "#eb6834", "#8a8578"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "text.parse_math": False})

V = SCORES[short(VERIFIER)]
SPECIMEN = None

if SPECIMEN is None:
    disagree = [i for i, (g, w) in enumerate(zip(groups, winners))
                if w is not None and g["labels"][V[i].argmax()] not in (None, w)]
    verifier_right = [i for i in disagree if C[i, V[i].argmax()] and not VOTE[i]]
    pool = verifier_right or disagree
    assert pool, "no problem where the vote and the verifier keep different answers"
    SPECIMEN = int(np.random.default_rng(SEED).choice([i for i in pool if len(groups[i]["reps"]) <= 6] or pool))

g, s, pick = groups[SPECIMEN], V[SPECIMEN], V[SPECIMEN].argmax()
print(problems[SPECIMEN]["problem"])
print(f"\nreference answer: {problems[SPECIMEN]['answer']}")

rows = sorted(set(g["labels"]), key=lambda l: (l is None, -g["labels"].count(l)))
fig, ax = plt.subplots(figsize=(10, 0.65 * len(rows) + 1.8))
rng = np.random.default_rng(SEED)
for y, label in enumerate(rows):
    idx = [k for k in range(N_CHAINS) if g["labels"][k] == label]
    ys = y + rng.uniform(-0.15, 0.15, len(idx))
    ax.scatter(s[idx], ys, s=80, c=[GREEN if C[SPECIMEN, k] else BLUE for k in idx],
               edgecolors="white", linewidths=1.5, zorder=3)
    if pick in idx:
        ax.scatter(s[pick], ys[idx.index(pick)], s=320, facecolors="none",
                   edgecolors=ORANGE, linewidths=2.5, zorder=4)


def row_name(label):
    answer = "(no answer)" if label is None else g["reps"][label]
    answer = answer if len(answer) <= 28 else answer[:27] + "…"
    vote = "   ← vote" if label is not None and label == winners[SPECIMEN] else ""
    return f"{answer}   ×{g['labels'].count(label)}{vote}"


ax.set_yticks(range(len(rows)), [row_name(l) for l in rows])
ax.set_ylim(len(rows) - 0.5, -0.5)
ax.set_xlabel(f"{short(VERIFIER)} score")
ax.grid(axis="x", alpha=0.3)
ax.scatter([], [], s=80, c=GREEN, label="right chain")
ax.scatter([], [], s=80, c=BLUE, label="wrong chain")
ax.scatter([], [], s=200, facecolors="none", edgecolors=ORANGE, linewidths=2.5, label="the verifier's pick")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
ax.set_title(f"Problem {SPECIMEN}: 16 chains, one row per answer", loc="left")
plt.tight_layout()
plt.show()

### Step 11: Problems Solved Against Chains Looked At

For each N from 1 to 16, every rule chooses from the same random subsets of N chains, `N_SUBSETS` per problem, and the point is the share of problems it solves, averaged over subsets. Best-of-N keeps the trained verifier's top chain, the vote keeps the most common answer (a tie is wrong), and the ceiling counts a problem as solved when any chain in the subset is right. One chain is the 3B's greedy chain from step 5, which does not change with N.

In [ ]:
Ns = np.arange(1, N_CHAINS + 1)
rng = np.random.default_rng(SEED)
SUBSETS = {n: np.array([rng.choice(N_CHAINS, n, replace=False) for _ in range(N_SUBSETS)]) for n in Ns}

curve = {"best": [], "vote": [], "ceiling": []}
for n in Ns:
    sub = SUBSETS[n]
    best, vote, ceiling = [], [], []
    for i, g in enumerate(groups):
        chosen = sub[np.arange(len(sub)), V[i][sub].argmax(axis=1)]
        best.append(C[i][chosen].mean())
        ceiling.append(C[i][sub].any(axis=1).mean())
        if not g["reps"]:
            vote.append(0.0)
            continue
        onehot = np.zeros((N_CHAINS, len(g["reps"])))
        for k, label in enumerate(g["labels"]):
            if label is not None:
                onehot[k, label] = 1
        counts = onehot[sub].sum(axis=1)                    # (subsets, answers)
        top = counts.max(axis=1)
        clear_winner = ((counts == top[:, None]).sum(axis=1) == 1) & (top > 0)
        vote.append((clear_winner & np.array(g["rep_correct"])[counts.argmax(axis=1)]).mean())
    curve["best"].append(100 * np.mean(best))
    curve["vote"].append(100 * np.mean(vote))
    curve["ceiling"].append(100 * np.mean(ceiling))

print(f"at N = 16: best-of-N {curve['best'][-1]:.1f}% (step 9: {100 * BEST[short(VERIFIER)].mean():.1f}%), "
      f"vote {curve['vote'][-1]:.1f}% (step 7: {100 * VOTE.mean():.1f}%)")

fig, ax = plt.subplots(figsize=(10, 5.5))
lines = [
    ("ceiling", curve["ceiling"], GREEN, "--"),
    (f"best-of-N, {short(VERIFIER)}", curve["best"], ORANGE, "-"),
    ("majority vote", curve["vote"], BLUE, "-"),
    ("one chain (greedy)", [100 * ONE_CHAIN.mean()] * len(Ns), GRAY, ":"),
]
for name, ys, color, style in lines:
    ax.plot(Ns, ys, color=color, linestyle=style, linewidth=2, marker="o" if style == "-" else None,
            markersize=4, label=name)
    ax.annotate(f"{ys[-1]:.0f}%", (Ns[-1], ys[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=9)
ax.set_xticks(Ns)
ax.set_xlabel("chains looked at (N)")
ax.set_ylabel("problems solved (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="lower right", frameon=False)
ax.set_title("Qwen2.5-3B-Instruct, MATH-500, chains at T = 0.3", loc="left")
plt.tight_layout()
plt.show()

### Step 12: Best-of-16 by Judge

One bar per judge: the problems Best-of-16 solves with that judge picking. Bars that clear the majority vote are orange. The grey lines mark one greedy chain (solid), the vote (dashed) and the ceiling (dotted).

In [ ]:
def judge_label(judge):
    return judge.removesuffix("-Instruct").replace("Skywork-o1-Open-PRM-Qwen-2.5-", "Skywork PRM ")


order = sorted(JUDGES, key=lambda j: BEST[j].mean())
values = [100 * BEST[j].mean() for j in order]
references = [
    ("one chain", 100 * ONE_CHAIN.mean(), "-"),
    ("majority vote", 100 * VOTE.mean(), "--"),
    ("ceiling", 100 * CEILING.mean(), ":"),
]

fig, ax = plt.subplots(figsize=(10, 0.6 * len(order) + 1.8))
colors = [ORANGE if BEST[j].mean() > VOTE.mean() else GRAY for j in order]
bars = ax.barh(range(len(order)), values, color=colors)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
for name, value, style in references:
    ax.axvline(value, color=GRAY, linestyle=style, linewidth=1.2, label=f"{name} ({value:.0f}%)")
ax.set_yticks(range(len(order)), [judge_label(j) for j in order])
ax.set_xlim(0, 100)
ax.set_xlabel("problems solved (%)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncols=3, frameon=False)
ax.set_title("Best-of-16, one bar per judge", loc="left")
plt.tight_layout()
plt.show()

### Step 13: Export for the Page

Two files, written to the Drive folder. Copy both to `web/data/`.

- `page-08-judges.json`: one Best-of-16 per judge.
- `page-08-pile.json`: every problem's 16 chains in the trained verifier's order, the three methods, the verifier's misses, and one miss drawn at random from those the vote also got wrong.

In [ ]:
KIND = {short(m): "language model" for m in GENERATORS} | {short(m): "reward model" for m in REWARD_MODELS}

site = {
    "placeholder": False,
    "writer": WRITER,
    "benchmark": "MATH-500",
    "n_problems": len(problems),
    "n_chains": N_CHAINS,
    "temperature": TEMPERATURE,
    "one_chain": float(ONE_CHAIN.mean()),
    "vote": float(VOTE.mean()),
    "ceiling": float(CEILING.mean()),
    "judges": [
        {"id": j, "label": judge_label(j), "kind": KIND[j], "best_of_n": float(BEST[j].mean())}
        for j in JUDGES
    ],
}
with open(saved("page-08-judges.json"), "w") as f:
    json.dump(site, f, indent=2)
print(json.dumps(site, indent=2))

In [ ]:
def mcnemar(a, b):
    """Exact two-sided test over the problems where a and b disagree."""
    x, y = int((a & ~b).sum()), int((b & ~a).sum())
    n = x + y
    if n == 0:
        return x, y, 1.0
    return x, y, min(1.0, 2 * sum(math.comb(n, i) for i in range(max(x, y), n + 1)) / 2 ** n)


V = SCORES[short(VERIFIER)]
KEPT = BEST[short(VERIFIER)]


def verifier_order(k):
    """Problem k's chains, highest score first; a tie goes to the earlier chain, as in `best_of`."""
    return sorted(range(N_CHAINS), key=lambda i: (-float(V[k, i]), i))


pile_rows = []
for k, (p, w) in enumerate(zip(problems, winners)):
    pile_rows.append({
        "id": p["unique_id"],
        "level": int(p["level"]),
        "n_correct": int(C[k].sum()),
        "vote": "tie" if w is None else "right" if VOTE[k] else "wrong",
        "greedy": bool(ONE_CHAIN[k]),
        "chains": [{"i": i, "c": bool(C[k, i]), "s": round(float(V[k, i]), 4)} for i in verifier_order(k)],
    })
pile_rows.sort(key=lambda r: (-r["n_correct"], r["id"]))

misses = [k for k in range(len(problems)) if CEILING[k] and not KEPT[k]]
pool = [k for k in misses if not VOTE[k]]
assert pool, "no verifier miss the vote also got wrong"
k = int(np.random.default_rng(SEED).choice(pool))
ranked = verifier_order(k)
top_right = next(i for i in ranked if C[k, i])
shown = ranked[:4] if top_right in ranked[:3] else ranked[:3] + [top_right]

best_only, vote_only, p_best_vote = mcnemar(KEPT, VOTE)
best_vs_one, one_vs_best, p_best_one = mcnemar(KEPT, ONE_CHAIN)
vote_vs_one, one_vs_vote, p_vote_one = mcnemar(VOTE, ONE_CHAIN)

pile = {
    "source": {"stand_in": False, "notebook": "8_Best_of_N.ipynb"},
    "writer": WRITER,
    "verifier": VERIFIER,
    "benchmark": "MATH-500",
    "levels": sorted({r["level"] for r in pile_rows}),
    "temperature": TEMPERATURE,
    "n_chains": N_CHAINS,
    "bars": {
        "one_chain": float(ONE_CHAIN.mean()),
        "chain_average": float(C.mean()),
        "vote": float(VOTE.mean()),
        "best_of_n": float(KEPT.mean()),
        "ceiling": float(CEILING.mean()),
        "best_vs_vote": {"best_only": best_only, "vote_only": vote_only, "p": p_best_vote},
        "best_vs_one": {"best_only": best_vs_one, "one_only": one_vs_best, "p": p_best_one},
        "vote_vs_one": {"vote_only": vote_vs_one, "one_only": one_vs_vote, "p": p_vote_one},
    },
    "misses": {"n": len(misses), "of": int(CEILING.sum()), "ids": [problems[m]["unique_id"] for m in misses]},
    "specimen": {
        "id": problems[k]["unique_id"],
        "question": problems[k]["problem"],
        "answer": problems[k]["answer"],
        "vote": "tie" if winners[k] is None else "wrong",
        "vote_answer": None if winners[k] is None else groups[k]["reps"][winners[k]],
        "n_correct": int(C[k].sum()),
        "right_rank": ranked.index(top_right) + 1,
        "cards": [{"i": i, "c": bool(C[k, i]), "s": round(float(V[k, i]), 4), "r": round(100 * float(V[k, i])),
                   "a": chains[k][i]["answer"], "trace": chains[k][i]["text"]} for i in shown],
    },
    "rows": pile_rows,
}
with open(saved("page-08-pile.json"), "w") as f:
    json.dump(pile, f, ensure_ascii=False, separators=(",", ":"))
print(f"page-08-pile.json: {os.path.getsize(saved('page-08-pile.json')) / 1e3:.0f} kB")
print(f"one chain {ONE_CHAIN.mean():.1%}, vote {VOTE.mean():.1%}, Best-of-16 {KEPT.mean():.1%}, ceiling {CEILING.mean():.1%}")
print(f"verifier misses: {len(misses)} of {int(CEILING.sum())}; Best-of-16 alone {best_only}, vote alone {vote_only}, p = {p_best_vote:.3f}")
